In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y bcftools > /dev/null
!bcftools --version | head -1


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
bcftools 1.19


In [3]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

Saving PGS000902_hmPOS_GRCh37.txt to PGS000902_hmPOS_GRCh37 (1).txt
['PGS000902_hmPOS_GRCh37 (1).txt']


In [4]:
import pandas as pd, numpy as np

SCORE_FILE = "PGS000902_hmPOS_GRCh37.txt.gz"   # <-- your exact file name

sc = pd.read_csv(SCORE_FILE, sep="\t", comment="#", low_memory=False)
print("Columns:", sc.columns.tolist())
print("Rows in score file:", len(sc))

sc = sc.dropna(subset=["hm_chr", "hm_pos"]).copy()
sc["chr"] = sc["hm_chr"].astype(str).str.replace("chr", "", regex=False)
sc["pos"] = sc["hm_pos"].astype(int)
sc["vid"] = sc["chr"] + ":" + sc["pos"].astype(str)   # variant ID used in all files
print("Variants with a GRCh37 position:", len(sc))
print("Chromosomes:", sorted(sc["chr"].unique(), key=lambda x: (len(x), x)))

sc[["chr", "pos"]].to_csv("regions.tsv", sep="\t", header=False, index=False)

meta_ad = pd.read_csv("meta.csv")
meta_ad["sample_id"].to_csv("samples.txt", index=False, header=False)
print("Samples:", len(meta_ad))

FileNotFoundError: [Errno 2] No such file or directory: 'PGS000902_hmPOS_GRCh37.txt.gz'

In [5]:
!wget -q https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS000902/ScoringFiles/Harmonized/PGS000902_hmPOS_GRCh37.txt.gz
!ls -lh PGS000902*

-rw-r--r-- 1 root root 6.4K Oct 29 11:10 'PGS000902_hmPOS_GRCh37 (1).txt'
-rw-r--r-- 1 root root 6.4K Oct 29 11:08  PGS000902_hmPOS_GRCh37.txt
-rw-r--r-- 1 root root 2.4K Aug  4  2022  PGS000902_hmPOS_GRCh37.txt.gz
-rw-r--r-- 1 root root   63 Oct 29 11:09 'PGS000902_hmPOS_GRCh37.txt.gz (1).md5'
-rw-r--r-- 1 root root   63 Oct 29 11:08  PGS000902_hmPOS_GRCh37.txt.gz.md5


In [8]:
from google.colab import files
up = files.upload()                      # choose meta.csv from module6_prs\data\
name = list(up)[0]
import os
if name != "meta.csv":
    os.rename(name, "meta.csv")          # fixes names like "meta (1).csv"
print(os.path.exists("meta.csv"))

Saving meta.csv to meta.csv
True


In [9]:
import pandas as pd, numpy as np

SCORE_FILE = "PGS000902_hmPOS_GRCh37.txt.gz"   # <-- your exact file name

sc = pd.read_csv(SCORE_FILE, sep="\t", comment="#", low_memory=False)
print("Columns:", sc.columns.tolist())
print("Rows in score file:", len(sc))

sc = sc.dropna(subset=["hm_chr", "hm_pos"]).copy()
sc["chr"] = sc["hm_chr"].astype(str).str.replace("chr", "", regex=False)
sc["pos"] = sc["hm_pos"].astype(int)
sc["vid"] = sc["chr"] + ":" + sc["pos"].astype(str)   # variant ID used in all files
print("Variants with a GRCh37 position:", len(sc))
print("Chromosomes:", sorted(sc["chr"].unique(), key=lambda x: (len(x), x)))

sc[["chr", "pos"]].to_csv("regions.tsv", sep="\t", header=False, index=False)

meta_ad = pd.read_csv("meta.csv")
meta_ad["sample_id"].to_csv("samples.txt", index=False, header=False)
print("Samples:", len(meta_ad))

Columns: ['rsID', 'chr_name', 'chr_position', 'effect_allele', 'effect_weight', 'hm_source', 'hm_rsID', 'hm_chr', 'hm_pos', 'hm_inferOtherAllele']
Rows in score file: 90
Variants with a GRCh37 position: 90
Chromosomes: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21']
Samples: 2336


In [12]:
!apt-get update -qq
!apt-get install -y bcftools
!which bcftools && bcftools --version | head -1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libhts3t64 libhtscodecs2
Suggested packages:
  python3-matplotlib texlive-latex-recommended
The following NEW packages will be installed:
  bcftools libhts3t64 libhtscodecs2
0 upgraded, 3 newly installed, 0 to remove and 5 not upgraded.
Need to get 1,326 kB of archives.
After this operation, 3,956 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 libhtscodecs2 amd64 1.6.0-1build1 [98.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 libhts3t64 amd64 1.19+ds-1.1build3 [437 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 bcftools amd64 1.19-1build2 [791 kB]
Fetched 1,326

In [13]:
import os
for f in ["regions.tsv", "samples.txt", "meta.csv", "PGS000902_hmPOS_GRCh37.txt.gz"]:
    print(f, os.path.exists(f))

regions.tsv True
samples.txt True
meta.csv True
PGS000902_hmPOS_GRCh37.txt.gz True


In [14]:
import subprocess

BASE = "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/"
chroms = sorted(sc["chr"].unique(), key=lambda x: int(x))   # autosomes only

for c in chroms:
    url = f"{BASE}ALL.chr{c}.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"
    cmd = (f"bcftools view -R regions.tsv -S samples.txt -v snps -m2 -M2 -Ou '{url}' | "
           f"bcftools query -H -f '%CHROM\\t%POS\\t%REF\\t%ALT[\\t%GT]\\n' > chr{c}.tsv")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    n = sum(1 for _ in open(f"chr{c}.tsv")) - 1
    print(f"chr{c}: {n} records", "| WARN: " + r.stderr[:150] if r.returncode else "")

chr1: 9 records 
chr2: 5 records 
chr3: 7 records 
chr4: 11 records 
chr5: 3 records 
chr6: 6 records 
chr7: 2 records 
chr8: 4 records 
chr9: 3 records 
chr10: 4 records 
chr11: 3 records 
chr12: 5 records 
chr13: 2 records 
chr14: 4 records 
chr15: 1 records 
chr16: 6 records 
chr17: 9 records 
chr18: 3 records 
chr19: 1 records 
chr20: 1 records 
chr21: 1 records 


In [15]:
frames = []
for c in chroms:
    d = pd.read_csv(f"chr{c}.tsv", sep="\t", dtype=str)
    d.columns = [col.lstrip("# ").split("]", 1)[-1].replace(":GT", "") for col in d.columns]
    frames.append(d)
vcf = pd.concat(frames, ignore_index=True)
vcf["vid"] = vcf["CHROM"] + ":" + vcf["POS"]
samples = list(meta_ad["sample_id"])

# Pick the score's other allele (published, or the harmonizer's inference)
def other_options(row):
    for col in ["other_allele", "hm_inferOtherAllele"]:
        if col in row and pd.notna(row[col]):
            return str(row[col]).upper().split("/")
    return []

keep_rows, score_rows, missing = [], [], []
for _, s in sc.iterrows():
    hits = vcf[vcf["vid"] == s["vid"]]
    eff = str(s["effect_allele"]).upper()
    match = None
    for _, h in hits.iterrows():
        alleles = {h["REF"], h["ALT"]}
        if eff in alleles:
            other = (alleles - {eff}).pop()
            if not other_options(s) or other in other_options(s):
                match = h; break
    if match is None:
        missing.append(s["vid"]); continue
    keep_rows.append(match)
    score_rows.append({"variant": s["vid"], "effect_allele": eff,
                       "other_allele": (set([match["REF"], match["ALT"]]) - {eff}).pop(),
                       "weight": float(s["effect_weight"])})

matched = pd.DataFrame(keep_rows).drop_duplicates("vid")
print(f"Score variants: {len(sc)} | found and allele-matched: {len(matched)} | missing: {len(missing)}")
print("Missing:", missing)

gt_map = {"0|0": 0, "0|1": 1, "1|0": 1, "1|1": 2}
dos = matched.set_index("vid")[samples].replace(gt_map)
bad = ~dos.isin([0, 1, 2])
print("Unexpected genotype calls:", int(bad.values.sum()))

Score variants: 90 | found and allele-matched: 90 | missing: 0
Missing: []


/tmp/ipykernel_685/339858106.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dos = matched.set_index("vid")[samples].replace(gt_map)


Unexpected genotype calls: 0


In [16]:
geno = dos.T.reset_index().rename(columns={"index": "sample_id"})
geno = geno.set_index("sample_id").loc[samples].reset_index()   # same row order as meta.csv
geno.to_csv("genotypes.csv", index=False)

matched[["vid", "REF", "ALT"]].rename(columns={"vid": "variant", "REF": "ref", "ALT": "alt"}) \
       .to_csv("variants.csv", index=False)
pd.DataFrame(score_rows).drop_duplicates("variant").to_csv("score.csv", index=False)

print(geno.shape)   # expect (2336, 1 + number of matched variants)

!zip -q pd_data.zip genotypes.csv variants.csv score.csv
files.download("pd_data.zip")

(2336, 91)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

geno  = pd.read_csv("genotypes.csv")
var   = pd.read_csv("variants.csv").set_index("variant")
score = pd.read_csv("score.csv")
meta  = pd.read_csv("meta.csv")[["sample_id", "ancestry", "age"]]

assert (geno["sample_id"].values == meta["sample_id"].values).all(), "row order mismatch"

G = geno[score["variant"]].to_numpy(float)
eff_is_alt = (score["effect_allele"].values == var.loc[score["variant"], "alt"].values)
D = np.where(eff_is_alt, G, 2 - G)

g = D @ score["weight"].values
g = (g - g.mean()) / g.std()
age_term = 0.35 * (meta["age"].values - 60) / 8
eur = meta["ancestry"].values == "EUR"
print("Effect allele = ALT for", int(eff_is_alt.sum()), "of", len(eff_is_alt), "variants")

Effect allele = ALT for 52 of 90 variants


In [18]:
def simulate(noise_sd, seed=6900):
    rng = np.random.default_rng(seed)
    liability = g + age_term + rng.normal(0, noise_sd, len(g))
    return (liability >= np.quantile(liability, 0.70)).astype(int)

for sd in [1.5, 2.0, 2.5, 3.0, 3.5]:
    y = simulate(sd)
    print(f"noise SD {sd}: EUR AUC = {roc_auc_score(y[eur], g[eur]):.3f}, cases = {y.mean():.0%}")

noise SD 1.5: EUR AUC = 0.707, cases = 30%
noise SD 2.0: EUR AUC = 0.641, cases = 30%
noise SD 2.5: EUR AUC = 0.623, cases = 30%
noise SD 3.0: EUR AUC = 0.610, cases = 30%
noise SD 3.5: EUR AUC = 0.601, cases = 30%


In [19]:
NOISE_SD = 2.0          # <-- change to the value you chose

meta["phenotype"] = simulate(NOISE_SD)
meta.to_csv("meta_pd.csv", index=False)
print(meta.groupby("ancestry")["phenotype"].mean().round(3))
print("Overall cases:", round(meta["phenotype"].mean(), 3))
files.download("meta_pd.csv")

ancestry
AFR    0.124
AMR    0.300
EAS    0.390
EUR    0.324
SAS    0.364
Name: phenotype, dtype: float64
Overall cases: 0.3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>